In [1]:
import pandas as pd
import time
import re
import random
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [2]:
def init_driver(headless=True):
    """Инициализация WebDriver"""
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    options.add_argument('--disable-blink-features=AutomationControlled')
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    wait = WebDriverWait(driver, 15)
    return driver, wait

In [3]:
SEEN_URLS = set()

In [4]:
def init_driver(headless=True):
    """Инициализация WebDriver"""
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    options.add_argument('--disable-blink-features=AutomationControlled')
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    wait = WebDriverWait(driver, 15)
    return driver, wait

In [5]:
def parse_calendar_page(driver, wait, calendar_url, department_filter="Modern+and+Contemporary+Art"):
    """Парсит страницу календаря с прокруткой для загрузки всех аукционов"""
    auction_data = []
    
    url = f"{calendar_url}?Departments={department_filter}"
    print(f"Календарь: {url}")
    driver.get(url)
    time.sleep(3)
    
    # === ПРОКРУТКА СТРАНИЦЫ ДЛЯ ЗАГРУЗКИ ВСЕХ АУКЦИОНОВ ===
    print("🔄 Прокручиваю страницу для загрузки всех элементов...")
    last_height = driver.execute_script("return document.body.scrollHeight")
    scroll_pause_time = 2  # секунды между прокрутками
    max_scrolls = 10  # максимальное число прокруток
    scroll_count = 0
    no_new_content_count = 0
    
    while scroll_count < max_scrolls:
        # Прокручиваем вниз
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_pause_time)
        
        # Считаем новую высоту
        new_height = driver.execute_script("return document.body.scrollHeight")
        
        # Проверяем, загрузилось ли что-то новое
        if new_height == last_height:
            no_new_content_count += 1
            if no_new_content_count >= 2:  # если 2 раза подряд ничего не загрузилось — выходим
                print("Все элементы загружены")
                break
        else:
            no_new_content_count = 0
        
        last_height = new_height
        scroll_count += 1
        print(f"  Прокрутка {scroll_count}/{max_scrolls} — высота: {new_height}px")
    
    # === ПАРСИНГ АУКЦИОНОВ ===
    try:
        auction_cards = driver.find_elements(By.CSS_SELECTOR, ".pah-auction-item.pah-calendar-page__auction-item")
        print(f"🎯 Найдено элементов: {len(auction_cards)}")
        
        # Парсим карточки аукционов
        for i, card in enumerate(auction_cards):
            try:
                # === Извлекаем заголовок ===
                title_elem = card.find_element(By.CSS_SELECTOR, "h3, h4, [class*='title'], .event-title")
                title = title_elem.text.strip()
                # === Извлекаем ссылку ===
                link_elem = card.find_element(By.XPATH, ".//a[@href]")
                auction_url = link_elem.get_attribute('href')
                    
                    # Пропускаем дубли
                if auction_url in SEEN_URLS:
                    continue
                SEEN_URLS.add(auction_url)
                
                # === Извлекаем дату и локацию ===
                date_text = 'Unknown'
                
                
                
                record = {
                    'auction_title': title,
                    'auction_url': auction_url,
                    'location': location_text,
                    'sale_date': date_text
                }
                auction_data.append(record)
                print(f"{title} | {location_text} | {date_text}")
                
            except Exception as e:
                print(f"Ошибка карточки {i+1}: {e}")
                continue
                    
    except Exception as e:
        print(f"Ошибка при поиске аукционов: {e}")
    
    return auction_data

In [6]:
def save_to_csv(auction_data, filename='phillips_auctions.csv'):
    """Сохраняет данные в CSV файл"""
    try:
        df = pd.DataFrame(auction_data)
        
        # Создаем колонки в правильном порядке
        columns_order = [
            'auction_title', 
            'auction_url', 
            'location', 
            'sale_date'
        ]
        
        # Добавляем только существующие колонки
        existing_columns = [col for col in columns_order if col in df.columns]
        df = df[existing_columns]
        
        # Удаляем дубли по URL
        df.drop_duplicates(subset=['auction_url'], inplace=True)
        
        # Сохраняем в CSV
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\nОбщая таблица сохранена: {filename}")
        print(f"Всего аукционов: {len(df)}")
        
        return df
        
    except Exception as e:
        print(f"Ошибка сохранения CSV: {e}")
        return None

In [7]:
def append_to_main_csv(auction_data, main_filename='phillips_all_auctions.csv'):
    """Добавляет данные в основной CSV файл"""
    try:
        df = pd.DataFrame(auction_data)
        
        if os.path.exists(main_filename):
            existing_df = pd.read_csv(main_filename)
            df = pd.concat([existing_df, df], ignore_index=True)
        
        df.drop_duplicates(subset=['auction_url'], inplace=True)
        df.to_csv(main_filename, index=False, encoding='utf-8-sig')
        print(f"Данные добавлены в {main_filename}")
        
    except Exception as e:
        print(f"Ошибка добавления в основной CSV: {e}")

In [8]:
if __name__ == "__main__":
    CALENDAR_URL = "https://www.phillips.com/calendar/results"
    DEPARTMENT_FILTER = "Modern+and+Contemporary+Art"
    
    print("Запуск парсинга аукционов Phillips...")
    print("="*60)
    
    driver, wait = init_driver(headless=False)  # False чтобы видеть процесс
    
    try:
        # Парсим календарь (один раз, без цикла по месяцам)
        auction_data = parse_calendar_page(
            driver, wait, 
            calendar_url=CALENDAR_URL,
            department_filter=DEPARTMENT_FILTER
        )
        
        # Сохраняем в CSV
        if auction_data:
            df = save_to_csv(auction_data, 'phillips_auctions.csv')
            
            # Предпросмотр
            if df is not None and not df.empty:
                print("\n" + "="*60)
                print("📋 ПРЕДПРОСМОТР")
                print("="*60)
                print(df.head(10).to_string(index=False))
        else:
            print("⚠️ Данные не собраны — проверьте селекторы или доступ к сайту")
            
    finally:
        driver.quit()
        print("\nБраузер закрыт")

🚀 Запуск парсинга аукционов Phillips...
📂 Открываю календарь: https://www.phillips.com/calendar/results?Departments=Modern+and+Contemporary+Art
🔄 Прокручиваю страницу для загрузки всех элементов...
  Прокрутка 1/10 — высота: 14910px
  Прокрутка 2/10 — высота: 23916px
  Прокрутка 3/10 — высота: 34858px
  Прокрутка 4/10 — высота: 45282px
  Прокрутка 5/10 — высота: 55989px
  Прокрутка 6/10 — высота: 66437px
  Прокрутка 7/10 — высота: 76342px
  Прокрутка 8/10 — высота: 86011px
  Прокрутка 9/10 — высота: 94877px
  Прокрутка 10/10 — высота: 104051px
🎯 Найдено элементов: 353
✅ MODERN & CONTEMPORARY ART | Unknown | Unknown
✅ MODERN & CONTEMPORARY ART EVENING SALE | Unknown | Unknown
✅ MODERN & CONTEMPORARY ART | Unknown | Unknown
✅ MODERN & CONTEMPORARY ART: ONLINE AUCTION, NEW YORK | Unknown | Unknown
✅ MODERN & CONTEMPORARY ART & DESIGN SALE | Unknown | Unknown
✅ NEW NOW: MODERN & CONTEMPORARY ART | Unknown | Unknown
✅ MODERN & CONTEMPORARY ART: ONLINE AUCTION, NEW YORK | Unknown | Unknown
✅